# Speed of Infection Visualization

Compares seeding models (and baselines) by how fast a contagion spreads
across the Nashville Meetup simplicial network. The y-axis is an
exponential-decay weighted score over the first 10 newly-infected nodes:

$$
\text{speed} \;=\; \mathbb{E}_{\text{trials}}\left[\; \sum_{i=1}^{10} e^{-\alpha\,t_i} \;\right]
$$

Two charts:
1. **speed vs. $\lambda$** (1-simplex infectivity), $\lambda_\Delta$ fixed
2. **speed vs. $\lambda_\Delta$** (2-simplex infectivity), $\lambda$ fixed

### Before running
Make sure `data/` has the processed graphs + probability lookup and
`weights/` has the model checkpoints you want to compare (see the
`MODEL_CHECKPOINTS` cell below).

## 1. Setup and data loading

In [ ]:
import os, sys, pickle, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, GATConv

# Make sure we're running from notebooks/ and that src/ is importable.
if os.path.basename(os.getcwd()) == 'models':
    os.chdir('..')
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

from contagion import MultiplexTopologyAdapter
from seeder import SimplicialSeeder
from speed_eval import (
    SpeedEvaluator,
    IterativeModelSeeder,
    SimplicialSeederAdapter,
    DegreeSeeder,
    RandomSeeder,
    load_model_checkpoint,
    plot_sweeps,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
DATA_DIR = 'data'

# Edge indices and node/event index maps
edge_simple = torch.load(os.path.join(DATA_DIR, 'edge_index_simple.pt'), weights_only=False)
edge_hyper  = torch.load(os.path.join(DATA_DIR, 'edge_index_hyper.pt'),  weights_only=False)
edge_attr_simple = torch.load(os.path.join(DATA_DIR, 'edge_attr_simple.pt'), weights_only=False)
edge_attr_hyper  = torch.load(os.path.join(DATA_DIR, 'edge_attr_hyper.pt'),  weights_only=False)

with open(os.path.join(DATA_DIR, 'user_idx.pkl'),  'rb') as f: user_idx     = pickle.load(f)
with open(os.path.join(DATA_DIR, 'event_idx.pkl'), 'rb') as f: idx_to_event = pickle.load(f)

adapter = MultiplexTopologyAdapter(edge_simple, edge_hyper, user_idx)
print(f'N = {adapter.N}, simple edges = {edge_simple.shape[1]}, triangles (rooted) = {len(adapter.triangles)}')

# Static graph dict consumed by all imitation models
df_user_feat = pd.read_csv(os.path.join(DATA_DIR, 'user_features.csv'), index_col=0)
df_user_feat = df_user_feat.apply(pd.to_numeric, errors='coerce')
ordered_member_ids = [user_idx[i] for i in range(adapter.N)]
aligned_user = df_user_feat.reindex(ordered_member_ids).fillna(0.0)
x_static = torch.tensor(aligned_user.values.astype(np.float32), dtype=torch.float)

static_graph = {
    'edge_index_1': edge_simple.to(device),
    'edge_attr_1':  edge_attr_simple.to(device),
    'edge_index_2': edge_hyper.to(device),
    'edge_attr_2':  edge_attr_hyper.to(device),
    'x_static':     x_static.to(device),
}

# Event features aligned to idx_to_event order
df_event_feat = pd.read_csv(os.path.join(DATA_DIR, 'event_features.csv'), index_col=0)
df_event_feat = df_event_feat.apply(pd.to_numeric, errors='coerce')
aligned_event = df_event_feat.reindex(idx_to_event).fillna(0.0)
event_feat_tensor = torch.tensor(aligned_event.values.astype(np.float32), dtype=torch.float)
event_dim = event_feat_tensor.shape[1]
node_dim = x_static.shape[1] + 1  # +1 seed-mask channel
print(f'node_dim = {node_dim}, event_dim = {event_dim}')

In [ ]:
# Probability lookup -> per-event susceptibility column.
# Same slicing trick as run_simulations.py so we don't keep the 1.9 GB table alive.
class CalibratedAffinityLookup:
    """Stub kept for pickle compatibility with run_simulations.py outputs."""
    pass

with open(os.path.join(DATA_DIR, 'probability_lookup.pkl'), 'rb') as f:
    prob_lookup = pickle.load(f)

# Pick a fixed sample of events for evaluation. 12 events -> ~12 runs per sweep
# value, enough to get a stable mean without blowing up wall time.
N_EVENTS = 12
rng = np.random.default_rng(seed=1234)
eval_event_ids = rng.choice(len(idx_to_event), size=N_EVENTS, replace=False).tolist()

ordered_user_ids = [user_idx[i] for i in range(len(user_idx))]
node_to_table_row = np.array(
    [prob_lookup.user_to_idx[uid] for uid in ordered_user_ids], dtype=np.int64
)

event_prob_columns = {}
event_feats = {}
for eid in eval_event_ids:
    original_event_id = idx_to_event[eid]
    col_idx = prob_lookup.event_to_idx[original_event_id]
    event_prob_columns[int(eid)] = np.ascontiguousarray(
        prob_lookup.probability_table[node_to_table_row, col_idx], dtype=np.float32
    )
    event_feats[int(eid)] = event_feat_tensor[eid]

del prob_lookup
gc.collect()
print(f'Prepared {len(eval_event_ids)} evaluation events.')

## 2. Model class definitions

The trainer assumes `forward(x_batch, static_graph, event_feat_batch) -> [B, N]`
logits. Re-defining the class bodies here lets `load_state_dict` work without
importing the notebooks. If you add a new model, paste its class definition
below and register it in the `MODEL_CHECKPOINTS` cell.

In [ ]:
class GAT_simple(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=1, edge_dim=1):
        super().__init__()
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads, edge_dim=edge_dim)
        self.gat2 = GATv2Conv(hidden_dim * heads, out_dim, heads=1, edge_dim=edge_dim)

    def forward(self, x_batch, static_graph, event_feat_batch):
        B, N, _ = x_batch.shape
        xs = []
        for b in range(B):
            x_b = x_batch[b]
            h = F.dropout(x_b, p=0.2, training=self.training)
            h = self.gat1(x_b, static_graph['edge_index_1'], static_graph['edge_attr_1'])
            h = F.elu(h)
            h = self.gat2(h, static_graph['edge_index_1'], static_graph['edge_attr_1'])
            xs.append(h)
        return torch.stack(xs).squeeze(2)


class GAT_hyper(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=1, edge_dim=1):
        super().__init__()
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads, edge_dim=edge_dim)
        self.gat2 = GATv2Conv(hidden_dim * heads, out_dim, heads=1, edge_dim=edge_dim)

    def forward(self, x_batch, static_graph, event_feat_batch):
        B, N, _ = x_batch.shape
        xs = []
        for b in range(B):
            x_b = x_batch[b]
            h = F.dropout(x_b, p=0.2, training=self.training)
            h = self.gat1(x_b, static_graph['edge_index_2'], static_graph['edge_attr_2'])
            h = F.elu(h)
            h = self.gat2(h, static_graph['edge_index_2'], static_graph['edge_attr_2'])
            xs.append(h)
        return torch.stack(xs).squeeze(2)


class GAT_simple_hyper(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=1, edge_dim=1):
        super().__init__()
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads, edge_dim=edge_dim)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim * heads, heads=heads, edge_dim=edge_dim)
        self.gat3 = GATv2Conv(hidden_dim * heads, out_dim, heads=1, edge_dim=edge_dim)

    def forward(self, x_batch, static_graph, event_feat_batch):
        B, N, _ = x_batch.shape
        xs = []
        for b in range(B):
            x_b = x_batch[b]
            h = F.dropout(x_b, p=0.2, training=self.training)
            h = self.gat1(x_b, static_graph['edge_index_1'], static_graph['edge_attr_1'])
            h = F.elu(h)
            h = F.dropout(h, p=0.2, training=self.training)
            h = self.gat2(h, static_graph['edge_index_2'], static_graph['edge_attr_2'])
            h = F.elu(h)
            h = self.gat2(h, static_graph['edge_index_1'], static_graph['edge_attr_1'])
            h = F.elu(h)
            h = self.gat3(h, static_graph['edge_index_2'], static_graph['edge_attr_2'])
            xs.append(h)
        return torch.stack(xs).squeeze(2)


class MultiplexSeedingModel(nn.Module):
    def __init__(self, node_dim, event_dim, edge_dim=1, hidden_dim=128):
        super().__init__()
        self.node_encoder = nn.Linear(node_dim, hidden_dim)
        self.event_encoder = nn.Linear(event_dim, hidden_dim)
        self.conv1_1 = GATConv(hidden_dim, hidden_dim // 2, edge_dim=edge_dim)
        self.conv2_1 = GATConv(hidden_dim, hidden_dim // 2, edge_dim=edge_dim)
        self.fusion_layer = nn.Linear(hidden_dim, hidden_dim)
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_batch, static_graph, event_feat_batch):
        B, N, _ = x_batch.shape
        x_encoded = F.relu(self.node_encoder(x_batch))
        event_encoded = F.relu(self.event_encoder(event_feat_batch))
        xs = []
        for b in range(B):
            x_b = x_encoded[b]
            x_1 = F.elu(self.conv1_1(x_b, static_graph['edge_index_1'], static_graph['edge_attr_1']))
            x_2 = F.elu(self.conv2_1(x_b, static_graph['edge_index_2'], static_graph['edge_attr_2']))
            x_fused = torch.cat([x_1, x_2], dim=-1)
            xs.append(F.relu(self.fusion_layer(x_fused)))
        batch_x = torch.stack(xs)
        event_expanded = event_encoded.unsqueeze(1).expand(-1, N, -1)
        combined = torch.cat([batch_x, event_expanded], dim=-1)
        logits = self.scorer(combined).squeeze(-1)
        seed_mask = x_batch[:, :, 0].bool()
        return logits.masked_fill(seed_mask, float('-1e9'))

## 3. Model checkpoints

Point each entry at the `.pt` weight file you want to evaluate.
The factory returns a fresh model instance each time `load_model_checkpoint`
is called so we don't alias state across runs.

In [ ]:
MODEL_CHECKPOINTS = {
    # name: (factory, checkpoint_path)
    'GAT_simple':       (lambda: GAT_simple(in_dim=-1, hidden_dim=8, out_dim=1),
                         'weights/REPLACE_ME/best_imitation_model_epoch_X.pt'),
    'GAT_hyper':        (lambda: GAT_hyper(in_dim=-1, hidden_dim=8, out_dim=1),
                         'weights/REPLACE_ME/best_imitation_model_epoch_X.pt'),
    'GAT_simple_hyper': (lambda: GAT_simple_hyper(in_dim=-1, hidden_dim=8, out_dim=1),
                         'weights/REPLACE_ME/best_imitation_model_epoch_X.pt'),
    # 'MultiplexSeeding': (lambda: MultiplexSeedingModel(node_dim=node_dim, event_dim=event_dim),
    #                      'weights/REPLACE_ME/best_imitation_model_epoch_X.pt'),
}

sample_event_feat = next(iter(event_feats.values()))

loaded_models = {}
for name, (factory, ckpt_path) in MODEL_CHECKPOINTS.items():
    if not os.path.exists(ckpt_path):
        print(f'[skip] {name}: checkpoint not found at {ckpt_path}')
        continue
    model = factory()
    load_model_checkpoint(model, ckpt_path, static_graph, sample_event_feat, device=device)
    loaded_models[name] = model
    print(f'[ok]   {name} loaded from {ckpt_path}')

print(f'\n{len(loaded_models)} model(s) ready.')

## 4. Build selectors (ML + baselines)

In [ ]:
selectors = {}

# ML-based: iterative argmax over each model's logits
for name, model in loaded_models.items():
    selectors[name] = IterativeModelSeeder(model, static_graph, device=device)

# Baselines
simplicial_seeder = SimplicialSeeder(
    num_nodes=adapter.N,
    links=adapter.links,
    flat_triangles=adapter.triangles,
    top_n=30,
)
selectors['SimplicialSeeder'] = SimplicialSeederAdapter(simplicial_seeder, beta=0.1, beta_delta=0.2)
selectors['DegreeTopK']       = DegreeSeeder(adapter.links)
selectors['Random']           = RandomSeeder(adapter.N, rng=np.random.default_rng(42))

print('Selectors registered:', list(selectors.keys()))

## 5. Run the sweeps

In [ ]:
# Sweep configuration
K_SEEDS         = 2     # matches max_seeds_per_iter used during imitation training
DECAY           = 0.3
INFECTED_TARGET = 10
MAX_SIM_STEPS   = 50
NUM_MC_TRIALS   = 40    # per (selector, event, sweep value) -- tweak for variance

LAM_VALUES   = np.linspace(0.02, 0.40, 10)
LAM_D_VALUES = np.linspace(0.02, 0.40, 10)
FIXED_LAM    = 0.10
FIXED_LAM_D  = 0.20

evaluator = SpeedEvaluator(
    num_nodes=adapter.N,
    edge_index_1=edge_simple,
    triangles_list=adapter.triangles,
    event_prob_columns=event_prob_columns,
    event_feats=event_feats,
    k_seeds=K_SEEDS,
    num_mc_trials=NUM_MC_TRIALS,
    decay=DECAY,
    infected_target=INFECTED_TARGET,
    max_sim_steps=MAX_SIM_STEPS,
    device=device,
)

print('Sweeping lambda...')
results_lam = evaluator.run_sweep(
    selectors=selectors,
    axis='lam',
    values=LAM_VALUES,
    fixed_value=FIXED_LAM_D,
    event_ids=eval_event_ids,
)

print('\nSweeping lambda_delta...')
results_lam_d = evaluator.run_sweep(
    selectors=selectors,
    axis='lam_d',
    values=LAM_D_VALUES,
    fixed_value=FIXED_LAM,
    event_ids=eval_event_ids,
)

print('\nDone.')

## 6. Plot

In [ ]:
fig = plot_sweeps(
    results_lam=results_lam,
    results_lam_d=results_lam_d,
    fixed_lam=FIXED_LAM,
    fixed_lam_d=FIXED_LAM_D,
    decay=DECAY,
    save_path='speed_of_infection.png',
)